# Run MoneyPrinterTurbo on Google Colab

This notebook installs [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) in an isolated Python 3.11 environment and launches its WebUI. Colab runtimes are temporary, so generated files are removed when the runtime is reset.

## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

## 2. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [19]:
from pyngrok import ngrok

ngrok.kill()

print("ngrok authentication already configured")

ngrok authentication already configured


## 3. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys. To use OpenRouter free LLMs, select **OpenRouter**, enter your OpenRouter API key, and keep the default `minimax/minimax-m3:free` model or choose another model from https://openrouter.ai/models?variant=free.

In [20]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Sanity checks before starting
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"Repo directory does not exist: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"webui/Main.py not found in {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(PORT)
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")

MoneyPrinterTurbo is ready:
https://garment-octane-crayon.ngrok-free.dev
Server log: /content/moneyprinterturbo-webui.log


In [27]:
!/content/MoneyPrinterTurbo/.venv/bin/python -c "import edge_tts; print(edge_tts.__version__)"

7.3.1


In [28]:
!/content/MoneyPrinterTurbo/.venv/bin/python -m edge_tts --list-voices | grep -i "odia\|or-IN"

In [30]:
import requests

url = "https://ttsfree.in/v1/catalog/voices?capability=text_to_speech"

r = requests.get(url, timeout=30)

print("Status:", r.status_code)
print(r.text[:5000])

Status: 200
{"capability":"text_to_speech","voices":[{"id":"assamese_amit","language":"Assamese","speaker":"Amit","recommended":true,"gender":"male","native_language":"Assamese","preview_url":"assets/previews/voices/amit-assamese.mp3","preview_duration_s":4.8},{"id":"assamese_sita","language":"Assamese","speaker":"Sita","recommended":true,"gender":"female","native_language":"Assamese","preview_url":"assets/previews/voices/sita-assamese.mp3","preview_duration_s":4.0},{"id":"bengali_aditi","language":"Bengali","speaker":"Aditi","recommended":true,"gender":"female","native_language":"Bengali","preview_url":"assets/previews/voices/aditi-bengali.mp3","preview_duration_s":4.0},{"id":"bengali_arjun","language":"Bengali","speaker":"Arjun","recommended":true,"gender":"male","native_language":"Bengali","preview_url":"assets/previews/voices/arjun-bengali.mp3","preview_duration_s":3.8},{"id":"bodo_bikram","language":"Bodo","speaker":"Bikram","recommended":true,"gender":"male","native_language":"Bo

In [31]:
import requests

url = "https://ttsfree.in/v1/catalog/voices?capability=text_to_speech"

data = requests.get(url, timeout=30).json()

for voice in data["voices"]:
    if voice.get("id", "").lower() in ["debjani", "manas"] or \
           "debjani" in str(voice).lower() or "manas" in str(voice).lower():
                   print(voice)

{'id': 'odia_debjani', 'language': 'Odia', 'speaker': 'Debjani', 'recommended': True, 'gender': 'female', 'native_language': 'Odia', 'preview_url': 'assets/previews/voices/debjani-odia.mp3', 'preview_duration_s': 4.5}
{'id': 'odia_manas', 'language': 'Odia', 'speaker': 'Manas', 'recommended': True, 'gender': 'male', 'native_language': 'Odia', 'preview_url': 'assets/previews/voices/manas-odia.mp3', 'preview_duration_s': 4.3}


In [33]:
import requests; v=requests.get("https://ttsfree.in/v1/catalog/voices?capability=text_to_speech",timeout=30).json()["voices"]; [(print(x["speaker"],"|",x["id"],"|",x.get("preview_url"))) for x in v if x["id"] in ["odia_debjani","odia_manas"]]

Debjani | odia_debjani | assets/previews/voices/debjani-odia.mp3
Manas | odia_manas | assets/previews/voices/manas-odia.mp3


[None, None]

In [34]:
import requests; base="https://ttsfree.in/"; [(open("/content/"+n+".mp3","wb").write(requests.get(base+p,timeout=30).content),print(n,"downloaded")) for n,p in [("debjani","assets/previews/voices/debjani-odia.mp3"),("manas","assets/previews/voices/manas-odia.mp3")]]

debjani downloaded
manas downloaded


[(54587, None), (52393, None)]

In [35]:
from IPython.display import Audio, display

display(Audio("/content/debjani.mp3"))
display(Audio("/content/manas.mp3"))

In [36]:
!git clone https://github.com/vexyl-ai/vexyl-tts.git /content/vexyl-tts-test

Cloning into '/content/vexyl-tts-test'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 19), reused 44 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 57.07 KiB | 1.54 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [37]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [39]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.0Gi       8.3Gi       6.2Mi       2.7Gi        10Gi
Swap:             0B          0B          0B


In [40]:
!cd /content/vexyl-tts-test && chmod +x setup.sh && VEXYL_TTS_DEVICE=cpu ./setup.sh


╔══════════════════════════════════════════════════════╗
║  VEXYL-TTS Server — Setup                            ║
║  ai4bharat/indic-parler-tts                          ║
╚══════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [1/6] Checking Python 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓ Found: Python 3.13.18

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [2/6] Detecting GPU
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ! No NVIDIA GPU detected. Installing CPU-only PyTorch.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [3/6] Creating Python virtual environment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Error: Command '['/content/vexyl-tts-test/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.


In [41]:
!apt-get update -qq && apt-get install -y python3.11 python3.11-venv

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.11-minimal libpython3.11-stdlib python3.11-distutils
  python3.11-lib2to3 python3.11-minimal
Suggested packages:
  binfmt-support
The following NEW packages will be installed:
  libpython3.11-minimal libpython3.11-stdlib python3.11 python3.11-distutils
  python3.11-lib2to3 python3.11-minimal python3.11-venv
0 upgraded, 7 newly installed, 0 to remove and 10 not upgraded.
Need to get 8,363 kB of archives.
After this operation, 25.3 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 libpython3.11-minimal amd64 3.11.18-1+noble1 [839 kB]
Get:2 https://ppa.launchpadcontent.net/d

In [42]:
!cd /content/vexyl-tts-test && python3.11 -m venv venv

In [43]:
!cd /content/vexyl-tts-test && venv/bin/pip install --upgrade pip && venv/bin/pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 23.1 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 61.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [torchaudio]


In [44]:
!cd /content/vexyl-tts-test && venv/bin/pip install git+https://github.com/huggingface/parler-tts.git transformers websockets numpy soundfile

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-0exhxzox
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-0exhxzox
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-p3vjyff7/descript-audiotools_8ad4d284c8a841d695136f1a85449c9b
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-p3vjyff7/descript-audiotools_8ad4d284c8a841d695136f1a85449c9b
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) .

In [45]:
!cd /content/vexyl-tts-test && venv/bin/python -c "from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts'); AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts'); AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'parler_tts'


In [46]:
!cd /content/vexyl-tts-test && venv/bin/pip show parler_tts && venv/bin/python -c "import sys; print(sys.executable); import parler_tts; print('IMPORT OK')"

Name: parler_tts
Version: 0.2.2
Summary: Toolkit for using and training Parler-TTS, a high-quality text-to-speech model.
Home-page: 
Author: 
Author-email: 
License: 
Location: /content/vexyl-tts-test/venv/lib/python3.11/site-packages
Requires: descript-audio-codec, descript-audiotools, protobuf, sentencepiece, torch, transformers
Required-by: 
/content/vexyl-tts-test/venv/bin/python
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys; print(sys.executable); import parler_tts; print('IMPORT OK')
                                       ^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'parler_tts'


In [47]:
!find /content/vexyl-tts-test/venv/lib/python3.11/site-packages -maxdepth 2 -iname "*parler*" -print

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts
/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts/configuration_parler_tts.py
/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts/modeling_parler_tts.py
/content/vexyl-tts-test/venv/lib/python3.11/site-packages/training/run_parler_tts_training.py
/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts-0.2.2.dist-info


In [48]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import sys; sys.path.insert(0, '/content/vexyl-tts-test/venv/lib/python3.11/site-packages'); print(sys.path[0]); import parler_tts; print('IMPORT OK:', parler_tts.__file__)"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys; sys.path.insert(0, '/content/vexyl-tts-test/venv/lib/python3.11/site-packages'); print(sys.path[0]); import parler_tts; print('IMPORT OK:', parler_tts.__file__)
                                                                                                                     ^^^^^^^^^^^^^^^^^
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts/__init__.py", line 4, in <module>
    from transformers import AutoConfig, AutoModel
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/transformers/__init__.py", line 26, in <module>
    from . import dependency_versions_check
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/transformers/dependency_versions_check.py", line 16, in <module>
    from .utils.versions import require_version, require_version_core
  File "/content/vexyl-tts-test/ven

In [49]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import PIL; print('Pillow:', PIL.__version__); print('PIL path:', PIL.__file__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import PIL; print('Pillow:', PIL.__version__); print('PIL path:', PIL.__file__)
    ^^^^^^^^^^
ModuleNotFoundError: No module named 'PIL'


In [50]:
!cd /content/vexyl-tts-test && venv/bin/pip install -U Pillow

In [51]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import sys,importlib.util; print('Python:',sys.executable); print('PIL spec:',importlib.util.find_spec('PIL')); print('PIL.Image spec:',importlib.util.find_spec('PIL.Image'));"

Python: /content/vexyl-tts-test/venv/bin/python
PIL spec: None
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys,importlib.util; print('Python:',sys.executable); print('PIL spec:',importlib.util.find_spec('PIL')); print('PIL.Image spec:',importlib.util.find_spec('PIL.Image'));
                                                                                                                                            ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "<frozen importlib.util>", line 91, in find_spec
ModuleNotFoundError: No module named 'PIL'


In [52]:
!cd /content/vexyl-tts-test && venv/bin/pip install --force-reinstall --no-cache-dir Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 39.1 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0


In [53]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import PIL; from PIL import Image; print('Pillow:', PIL.__version__); print('PIL OK'); print('Image OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import PIL; from PIL import Image; print('Pillow:', PIL.__version__); print('PIL OK'); print('Image OK')
    ^^^^^^^^^^
ModuleNotFoundError: No module named 'PIL'


In [54]:
!cd /content/vexyl-tts-test && echo "=== PIL folder ===" && ls -la venv/lib/python3.11/site-packages/PIL 2>/dev/null | head -20 && echo "=== Pillow files ===" && venv/bin/pip show -f Pillow | head -40

=== PIL folder ===
total 5712
drwxr-xr-x   3 root root    4096 Nov  1 21:29 .
drwxr-xr-x 212 root root   12288 Nov  1 21:29 ..
-rwxr-xr-x   1 root root   97889 Nov  1 21:29 _avif.cpython-311-x86_64-linux-gnu.so
-rw-r--r--   1 root root    9419 Nov  1 21:29 AvifImagePlugin.py
-rw-r--r--   1 root root      63 Nov  1 21:29 _avif.pyi
-rw-r--r--   1 root root    3260 Nov  1 21:29 BdfFontFile.py
-rw-r--r--   1 root root    2613 Nov  1 21:29 _binary.py
-rw-r--r--   1 root root   16529 Nov  1 21:29 BlpImagePlugin.py
-rw-r--r--   1 root root   20391 Nov  1 21:29 BmpImagePlugin.py
-rw-r--r--   1 root root    1675 Nov  1 21:29 BufrStubImagePlugin.py
-rw-r--r--   1 root root    4647 Nov  1 21:29 ContainerIO.py
-rw-r--r--   1 root root    2747 Nov  1 21:29 CurImagePlugin.py
-rw-r--r--   1 root root    2318 Nov  1 21:29 DcxImagePlugin.py
-rw-r--r--   1 root root   19182 Nov  1 21:29 DdsImagePlugin.py
-rw-r--r--   1 root root    2067 Nov  1 21:29 _deprecate.py
-rw-r--r--   1 root root   17133 Nov  1 

In [55]:
!cd /content/vexyl-tts-test && ls -l venv/lib/python3.11/site-packages/PIL/__init__.py && venv/bin/python -c "import sys; print('\\n'.join(sys.path)); import PIL; print('PIL OK:', PIL.__file__)"

-rw-r--r-- 1 root root 2035 Nov  1 21:29 venv/lib/python3.11/site-packages/PIL/__init__.py

/env/python
/usr/lib/python313.zip
/usr/lib/python3.13
/usr/lib/python3.13/lib-dynload
/content/vexyl-tts-test/venv/lib/python3.13/site-packages
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys; print('\n'.join(sys.path)); import PIL; print('PIL OK:', PIL.__file__)
                                            ^^^^^^^^^^
ModuleNotFoundError: No module named 'PIL'


In [56]:
!cd /content/vexyl-tts-test && echo "venv python:" && readlink -f venv/bin/python && echo "version:" && venv/bin/python --version && echo "python3.11:" && venv/bin/python3.11 --version

venv python:
/usr/bin/python3.13
version:
Python 3.13.18
python3.11:
Python 3.11.18


In [1]:
!cd /content/vexyl-tts-test && rm -rf venv && python3.11 -m venv venv && venv/bin/python --version

/bin/bash: line 1: cd: /content/vexyl-tts-test: No such file or directory


In [2]:
!git clone https://github.com/vexyl-ai/vexyl-tts.git /content/vexyl-tts-test

Cloning into '/content/vexyl-tts-test'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 19), reused 44 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 57.07 KiB | 11.41 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
!cd /content/vexyl-tts-test && python3.11 -m venv venv && venv/bin/python --version

/bin/bash: line 1: python3.11: command not found


In [4]:
!apt-get update -qq && apt-get install -y python3.11 python3.11-venv

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.11-minimal libpython3.11-stdlib python3.11-distutils
  python3.11-lib2to3 python3.11-minimal
Suggested packages:
  binfmt-support
The following NEW packages will be installed:
  libpython3.11-minimal libpython3.11-stdlib python3.11 python3.11-distutils
  python3.11-lib2to3 python3.11-minimal python3.11-venv
0 upgraded, 7 newly installed, 0 to remove and 10 not upgraded.
Need to get 8,363 kB of archives.
After this operation, 25.3 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 libpython3.11-minimal amd64 3.11.18-1+noble1 [839 kB]
Get:2 https://ppa.launchpadcontent.net/d

In [5]:
!cd /content/vexyl-tts-test && python3.11 -m venv venv && venv/bin/python --version

Python 3.11.18


In [6]:
!cd /content/vexyl-tts-test && venv/bin/pip install --upgrade pip && venv/bin/pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 64.0 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 99.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [torchaudio]


In [7]:
!cd /content/vexyl-tts-test && venv/bin/pip install git+https://github.com/huggingface/parler-tts.git transformers websockets numpy soundfile

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-necvj4e_
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-necvj4e_
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-t57xzizx/descript-audiotools_2ff25312366b44bca842202c4be1e4da
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-t57xzizx/descript-audiotools_2ff25312366b44bca842202c4be1e4da
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) .

In [8]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import parler_tts; print('PARLER-TTS OK'); print(parler_tts.__file__)"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Flash attention 2 is not installed
PARLER-TTS OK
/content/vexyl-tts-test/venv/lib/python3.11/site-packages/parler_tts/__init__.py


In [9]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

model_id = "ai4bharat/indic-parler-tts"

print("Downloading/loading model...")
model = ParlerTTSForConditionalGeneration.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("MODEL LOADED OK")

ModuleNotFoundError: No module named 'parler_tts'

In [10]:
!cd /content/vexyl-tts-test && venv/bin/python -c "from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; print('Loading model...'); model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts'); print('Model loaded'); tokenizer=AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); print('MODEL + TOKENIZER OK')"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Flash attention 2 is not installed
Loading model...
Traceback (most recent call last):
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/huggingface_hub/utils/_http.py", line 403, in hf_raise_for_status
    response.raise_for_status()
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/requests/models.py", line 1167, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/ai4bharat/indic-parler-tts/resolve/main/config.json

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/vexyl-tts-test/venv/lib/python3.11/site-packages/transformers/utils/hub.py", line 403, in cached_file
    re

In [11]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token, add_to_git_credential=False)

print("✅ Hugging Face login successful")

✅ Hugging Face login successful


In [12]:
!cd /content/vexyl-tts-test && venv/bin/python -c "from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; print('Loading model...'); model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts'); print('Model loaded'); tokenizer=AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); print('MODEL + TOKENIZER OK')"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Flash attention 2 is not installed
Loading model...


Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "rela

In [13]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import torch, soundfile as sf; from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; device='cpu'; print('Loading...'); model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts').to(device); tokenizer=AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); description_tokenizer=AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path); prompt='ନମସ୍କାର! ଆପଣଙ୍କୁ ସ୍ୱାଗତ। ଆଜି ଆମେ ଓଡ଼ିଆ ଭାଷାରେ ଏକ ଛୋଟ ଶିକ୍ଷାମୂଳକ ଭିଡିଓ ତିଆରି କରିବା।'; description='Debjani speaks in a clear, natural and warm female voice at a moderate pace. The recording is very clear, with no background noise.'; d=description_tokenizer(description,return_tensors='pt').to(device); p=tokenizer(prompt,return_tensors='pt').to(device); print('Generating Debjani...'); audio=model.generate(input_ids=d.input_ids,attention_mask=d.attention_mask,prompt_input_ids=p.input_ids,prompt_attention_mask=p.attention_mask); sf.write('/content/Debjani_odia_test.wav',audio.cpu().numpy().squeeze(),model.config.sampling_rate); print('✅ Debjani sample created: /content/Debjani_odia_test.wav')"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Flash attention 2 is not installed
Loading...
Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_att

In [14]:
from IPython.display import Audio, display

display(Audio("/content/Debjani_odia_test.wav"))

In [15]:
!cd /content/vexyl-tts-test && venv/bin/python -c "import torch, soundfile as sf; from parler_tts import ParlerTTSForConditionalGeneration; from transformers import AutoTokenizer; device='cpu'; print('Loading model...'); model=ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts').to(device); tokenizer=AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts'); description_tokenizer=AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path); prompt='ନମସ୍କାର! ଆପଣଙ୍କୁ ସ୍ୱାଗତ। ଆଜି ଆମେ ଓଡ଼ିଆ ଭାଷାରେ ଏକ ଛୋଟ ଶିକ୍ଷାମୂଳକ ଭିଡିଓ ତିଆରି କରିବା।'; description='Manas speaks in a clear, natural and warm male voice at a moderate pace. The recording is very clear, with no background noise.'; d=description_tokenizer(description,return_tensors='pt').to(device); p=tokenizer(prompt,return_tensors='pt').to(device); print('Generating Manas...'); audio=model.generate(input_ids=d.input_ids,attention_mask=d.attention_mask,prompt_input_ids=p.input_ids,prompt_attention_mask=p.attention_mask); sf.write('/content/Manas_odia_test.wav',audio.cpu().numpy().squeeze(),model.config.sampling_rate); print('✅ Manas sample created: /content/Manas_odia_test.wav')"

/content/vexyl-tts-test/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Flash attention 2 is not installed
Loading model...
Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relati

In [16]:
from IPython.display import Audio, display

display(Audio("/content/Manas_odia_test.wav"))

In [17]:
!find /content -type f \( -name "*.mp4" -o -name "*.mov" -o -name "*.mkv" \) 2>/dev/null | head -20

In [18]:
!find /content -maxdepth 4 -type d \( -iname "*output*" -o -iname "*video*" \) 2>/dev/null | head -30

In [19]:
!find /content -maxdepth 2 -type d | head -40

/content
/content/.config
/content/.config/configurations
/content/.config/logs
/content/vexyl-tts-test
/content/vexyl-tts-test/venv
/content/vexyl-tts-test/.git
/content/sample_data


In [20]:
!find /content -maxdepth 3 -type f -iname "*moneyprinter*" 2>/dev/null